# 02 - EDA

## Objective

After completing the data audit, this notebook explores the dataset in greater depth to understand the behavior, usefulness, and business meaning of each feature.

The analysis will:

- examine numerical and categorical feature distributions;
- compare feature behavior across churn classes;
- identify potentially useful, redundant, weak, or suspicious features;
- investigate relationships between features and the target;
- detect patterns that may influence preprocessing and modeling decisions;
- document business observations and modeling implications.

No final transformations will be applied in this notebook. The conclusions from the EDA will guide the preprocessing stage, where features will be converted, encoded, scaled, removed, or otherwise transformed as required.

## Load audited data

The dataset is loaded for exploratory analysis after completing the initial data audit.

The previous audit identified the main data-quality issues, including invalid blank values in `TotalCharges`, the unique identifier `customerID`, target imbalance, and potential leakage risks.

This notebook will use the audited structure of the dataset to investigate feature distributions, feature-target relationships, and business patterns. Final preprocessing transformations will not be applied at this stage.

In [38]:
from pathlib import Path
import pandas as pd
import numpy as np
from numpy.testing.print_coercion_tables import print_new_cast_table

In [39]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_DIR / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {RAW_DATA_PATH.resolve()}")
dataset = pd.read_csv(RAW_DATA_PATH)

In [40]:
dataset.head(8)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No


### Validation

The dataset was loaded successfully with [number] rows and [number] columns.

The expected identifier, numerical, categorical, and target columns are present. The dataset is ready for exploratory analysis based on the findings documented in the data-audit notebook.

In [41]:
print(f'Columns from dataset: {dataset.columns.tolist()}\n'
      f'Len of columns {len(dataset.columns.tolist())}')

Columns from dataset: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
Len of columns 21


## Target Balance

This section examines whether the `Churn` target classes are balanced.

The result will guide later decisions about train-test splitting, model training, class imbalance techniques, and evaluation metrics.

In [58]:
target_counts = dataset["Churn"].value_counts(dropna=False)
percentage_no = (target_counts["No"] / (target_counts["No"] + target_counts["Yes"]))*100
percentage_yes = (target_counts["Yes"] / (target_counts["No"] + target_counts["Yes"]))*100
imbalance_ratio = target_counts["No"] / target_counts["Yes"]

print(target_counts)
print("-"*50)
print(f'Percentage for class No: {percentage_no:.2f}%')
print(f'Percentage for class Yes: {percentage_yes:.2f}%')
print("-"*50)
print(f'Imbalance ratio: {imbalance_ratio:.2f}')

Churn
No     5174
Yes    1869
Name: count, dtype: int64
--------------------------------------------------
Percentage for class No: 73.46%
Percentage for class Yes: 26.54%
--------------------------------------------------
Imbalance ratio: 2.77


### Interpretation

The target variable is moderately imbalanced.

The `No` class represents 73.46% of the observations, while the `Yes` class represents 26.54%. There are approximately 2.77 non-churn customers for every churn customer.

Because of this imbalance, accuracy alone may produce a misleading evaluation. The train-test split should preserve the class proportions using stratification, and the modeling stage should also evaluate recall, precision, F1-score, balanced accuracy, and PR-AUC for the churn class.

## Numerical feature list

In this section, I will analyze the numerical distributions and verify whether the values are valid and reasonable. I also need to convert `TotalCharges` to a numeric type before analyzing it.